<div style="text-align:center; padding:24px 0 12px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="200"/>
</div>


# GoogleAdsPulse — Paid Media Analytics
## Notebook 1 — Contexte métier & Vue d'ensemble des KPIs

---

> **Objectif du notebook 1 :** prendre en main le brief de Marc-Aurèle, découvrir les 5 tables
> Google Ads, auditer la qualité des données, et calculer les **10 KPIs Coupler.io de référence**
> qui alimenteront le dashboard final.

> **Ce notebook utilise uniquement pandas.** Le SQL analytique avancé (window functions,
> cohort analysis, détection d'anomalies) sera traité dans le **NB2**.

| | |
|---|---|
| **Projet** | GoogleAdsPulse — Paid Media Analytics |
| **Persona** | Marc-Aurèle TOURÉ, Paid Media Manager chez FluxData Agency |
| **Stack** | `pandas`, `matplotlib` |
| **Durée** | 2h30 à 3h30 |
| **Sortie** | 10 KPIs Coupler.io + viz Overview + liste des anomalies à corriger |


---
## 1. Contexte métier

### FluxData Agency — une agence digitale d'Abidjan

FluxData Agency est une agence digitale basée à Abidjan qui gère les comptes Google Ads
de **5 clients PME** d'Afrique de l'Ouest sur des secteurs variés : e-commerce
(TechShop CI), SaaS finance (BankAfrica SaaS), EdTech (FormaProPlus), hôtellerie
(AfriHotels Group) et santé B2B (MedSupply Pro). Budget cumulé géré : **~625 000 EUR**
sur 2023-2024.

### Le brief de Marc-Aurèle

> *« Chaque lundi, je passe 6 à 8 heures à consolider les reportings Google Ads de mes 5
> clients dans Excel. Je copie-colle les exports, je recalcule les deltas vs la semaine
> précédente, je refais les mêmes graphiques. C'est long, pas scalable, et mes clients
> veulent maintenant un accès direct à leurs KPIs.*
>
> *J'ai aussi du mal à savoir ce qui marche vraiment : Google Ads me donne un CPC flatteur
> sur certaines campagnes, mais quand je regarde le revenu généré, ce n'est pas forcément
> celles-là qui rapportent. Et avec 25 campagnes actives, je ne peux pas surveiller
> manuellement chaque dérapage budgétaire — il y a toujours une campagne qui consomme 3×
> son budget habituel sans que je le remarque avant la fin du mois.*
>
> *Il me faut un dashboard unifié qui centralise mes 5 comptes, affiche les KPIs avec leurs
> deltas, et détecte automatiquement les anomalies. »*

### 🎓 MÉTHODE — Traduire un brief verbal en questions analytiques

Marc-Aurèle exprime 3 préoccupations distinctes qu'il faut structurer en questions
analytiques concrètes :

| Phrase du brief | Question analytique | Livrable |
|---|---|---|
| *« Consolider les reportings de 5 clients chaque lundi »* | Centraliser les KPIs Google Ads standards dans une vue unifiée | NB1 — 10 KPIs Coupler.io |
| *« Google Ads me donne un CPC flatteur, mais le revenu ne suit pas »* | Quelles campagnes ont le **meilleur ROAS** (revenu / spend) ? | NB2 — RANK() + NTILE |
| *« Une campagne consomme 3× son budget sans que je le remarque »* | Détecter automatiquement les **dérapages budgétaires** | NB2 — Z-score SQL |

### 🏢 MÉTIER — Ce qui différencie un Paid Media Manager senior

**1. Savoir lire au-delà du CPC affiché**

Les plateformes publicitaires optimisent leurs métriques pour paraître bonnes. Un CPC bas
peut cacher un trafic non qualifié qui ne convertit pas. Les vrais KPIs de pilotage sont
**Cost/Conversion** et **ROAS** (Return On Ad Spend = Conversions Value / Cost).

**2. Arbitrer Brand vs Generic**

Les campagnes **Brand** (le client tape le nom de l'annonceur) ont des CPC ultra-bas et
des conversions élevées — mais c'est du trafic qui serait venu quand même par l'organique.
Les campagnes **Generic** (mots-clés métier) coûtent plus cher mais génèrent du *vrai*
trafic incrémental. Bien arbitrer les deux est un vrai enjeu.

**3. Détecter les dérapages avant qu'ils ne coûtent**

Avec 25 campagnes actives, surveiller à l'œil est impossible. Un système d'alerte
automatique (z-score sur le spend quotidien) permet de repérer les jours anormaux
sans surcharge cognitive.


---
## 2. Dictionnaire des données — 5 tables

### Schéma en étoile

```
   accounts ──────► campaigns ──────► ads ──────► keywords
                        │               │             │
                        └───────────────┴─────────────┘
                                        │
                                        ▼
                          performance_quotidienne (fact)
                          granularité : date × campaign × ad × keyword
                                       × device × country × hour
```

### Détail des 5 tables

**`accounts.csv` (5 lignes)** — Comptes Google Ads gérés par l'agence

| Colonne | Type | Description |
|---|---|---|
| `account_id` | str | ACC001 → ACC005 |
| `account_name` | str | Nom commercial du client |
| `currency` | str | EUR (unifié pour tous les comptes) |
| `industry` | str | E-commerce, SaaS Finance, EdTech, Travel, B2B Health |
| `manager` | str | Marc-Aurèle T. ou Aïssatou D. |

**`campaigns.csv` (~25 lignes)** — Campagnes Google Ads

| Colonne | Type | Description |
|---|---|---|
| `campaign_id` | str | CAMP0001 → CAMP00XX |
| `campaign_name` | str | Nom du type `Brand-Search-CI`, `PerfMax-LeadGen`, etc. |
| `campaign_type` | str | Search / Display / Performance Max / Shopping / Video |
| `bid_strategy` | str | Manual CPC / Target CPA / Target ROAS / Maximize Conversions / Maximize Clicks |
| `budget_quotidien_eur` | int | Budget journalier planifié |
| `statut` | str | Enabled / Paused / Removed |
| `objectif` | str | Sales / Leads / Website traffic / Brand awareness |

**`ads.csv` (~120 lignes)** — Annonces avec Quality Score

| Colonne | Type | Description |
|---|---|---|
| `ad_id` | str | AD00001 → AD00XXX |
| `ad_group_name` | str | Nom de l'ad group parent |
| `headline_1`, `headline_2`, `description` | str | Copy publicitaire |
| `quality_score` | int | 1 à 10 — score de pertinence Google |

**`keywords.csv` (~370 lignes)** — Mots-clés (Search/Shopping)

| Colonne | Type | Description |
|---|---|---|
| `keyword_id` | str | KW00001 → KW00XXX |
| `keyword_text` | str | Texte du mot-clé |
| `match_type` | str | Exact / Phrase / Broad |
| `quality_score` | int | 1 à 10 |

**`performance_quotidienne.csv` (~41 400 lignes)** — Table de faits granulaire

| Colonne | Type | Description |
|---|---|---|
| `date` | date | Date du snapshot quotidien |
| `account_id`, `campaign_id`, `ad_id`, `keyword_id` | str | FKs |
| `device` | str | Desktop / Mobile / Tablet |
| `country`, `day_of_week`, `hour` | mixte | Contexte temporel et géo |
| `impressions`, `clicks`, `cost_eur` | int/float | **3 métriques de base** |
| `conversions`, `conversion_value_eur` | int/float | **Résultat** |
| `conversion_type` | str | Purchase / Lead / Signup / Demo Request / Form Submit / Phone Call |
| `avg_position` | float | Position moyenne dans les SERPs |

### ⚠️ Points d'attention dès la lecture du dictionnaire

| Piège | Impact | Vérification |
|---|---|---|
| `keyword_id` NULL pour les campaigns Display/Video/PMax | Normal (automated) | LEFT JOIN obligatoire |
| `quality_score` doit être dans [1-10] | Scores hors plage = bug ETL | Diagnostic à faire |
| `clicks > impressions` : impossible logiquement | Bug d'attribution | Diagnostic à faire |
| `cost_eur` doit être ≥ 0 | Les cost négatifs = bug de refund | Diagnostic à faire |
| `conversions > clicks` | Rare mais possible (multi-touch) | À flagger |


---
## 3. Setup & chargement

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os, sys

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

# Palette DataProjectLab — GoogleAdsPulse (style Coupler.io)
COLORS = {
    'primary':   '#534AB7',  # Violet DPL
    'secondary': '#1D9E75',  # Vert
    'warning':   '#EF9F27',  # Orange
    'danger':    '#E24B4A',  # Rouge
    'neutral':   '#888780',
    'blue':      '#0EA5E9',  # Bleu Coupler
}
CAMPAIGN_COLORS = {
    'Search':          '#4285F4',  # Google blue
    'Shopping':        '#34A853',  # Google green
    'Performance Max': '#9333EA',  # Violet
    'Display':         '#FBBC04',  # Google yellow
    'Video':           '#EA4335',  # Google red
}

print(f'✅ Environnement prêt — pandas {pd.__version__}')


### Chargement depuis Google Drive ou GitHub

**Option A :** Drive si dataset partagé dans `Mon Drive/DataProjectLab/googleadspulse/data/`.
**Option B :** fallback GitHub automatique.


In [ ]:
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/DataProjectLab/googleadspulse/data/'
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/googleadspulse/outputs/'
else:
    DATA_PATH = './data/'
    SAVE_PATH = './outputs/'

os.makedirs(SAVE_PATH, exist_ok=True)
USE_GITHUB = not os.path.exists(DATA_PATH)
BASE_URL   = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse/data/'

def load_csv(name, **kwargs):
    path = DATA_PATH + name if not USE_GITHUB else BASE_URL + name
    return pd.read_csv(path, **kwargs)

print(f'📁 Source : {"GitHub" if USE_GITHUB else DATA_PATH}')


In [ ]:
# Chargement des 5 tables
accounts   = load_csv('accounts.csv', parse_dates=['date_onboarding'])
campaigns  = load_csv('campaigns.csv', parse_dates=['start_date', 'end_date'])
ads        = load_csv('ads.csv')
keywords   = load_csv('keywords.csv')
perf       = load_csv('performance_quotidienne.csv', parse_dates=['date'])

print(f"{'Table':<28} {'Lignes':>10} {'Colonnes':>10}")
print('-' * 52)
for name, df in [('accounts', accounts), ('campaigns', campaigns),
                  ('ads', ads), ('keywords', keywords),
                  ('performance_quotidienne', perf)]:
    print(f"{name:<28} {len(df):>10,} {len(df.columns):>10}")

print(f'\nPériode couverte : {perf["date"].min().date()} → {perf["date"].max().date()}')
print(f'Durée            : {(perf["date"].max() - perf["date"].min()).days + 1} jours')
print(f'Accounts         : {accounts["account_name"].tolist()}')


---
## 4. Exploration avec `pandas`

### 🎓 MÉTHODE — Routine d'exploration

Pour chaque table clé, 4 méthodes essentielles :

| Méthode | Révèle |
|---|---|
| `.head(n)` | Format visuel, structure brute |
| `.info()` | Types + nulls + mémoire |
| `.describe()` | Stats descriptives — détection d'outliers |
| `.value_counts()` | Distribution catégorielles — valeurs aberrantes |


In [ ]:
# ── Table performance_quotidienne — la plus importante ────────────────
print('=' * 60)
print('  EXPLORATION — performance_quotidienne (table de faits)')
print('=' * 60)

print('\n[.head(3)]')
print(perf.head(3).to_string())

print('\n[.info()]')
perf.info()


In [ ]:
# Statistiques des colonnes numériques clés
print('[.describe()] Métriques principales :')
perf[['impressions','clicks','cost_eur','conversions','conversion_value_eur','avg_position']].describe().round(2)


In [ ]:
# Distribution des catégorielles
print('Distribution DEVICE :')
print(perf['device'].value_counts())
print(f'\n% par device :')
print((perf['device'].value_counts(normalize=True)*100).round(1).astype(str) + ' %')

print('\nDistribution COUNTRY :')
print(perf['country'].value_counts())

print('\nDistribution CONVERSION_TYPE :')
print(perf['conversion_type'].value_counts())


In [ ]:
# ── Table campaigns ───────────────────────────────────────────────────
print('=' * 60)
print('  EXPLORATION — campaigns (25 lignes)')
print('=' * 60)
print('\nDistribution par CAMPAIGN_TYPE :')
print(campaigns['campaign_type'].value_counts())

print('\nDistribution par STATUT :')
print(campaigns['statut'].value_counts())

print('\nDistribution par OBJECTIF :')
print(campaigns['objectif'].value_counts())

print('\nBudget total planifié (journalier cumulé) :')
print(f'  {campaigns["budget_quotidien_eur"].sum():,} EUR/jour')


In [ ]:
# ── Table keywords ────────────────────────────────────────────────────
print('=' * 60)
print('  EXPLORATION — keywords')
print('=' * 60)
print(f'\nTotal keywords : {len(keywords):,}')

print('\nDistribution MATCH_TYPE :')
print(keywords['match_type'].value_counts())

print('\nDistribution QUALITY_SCORE :')
print(keywords['quality_score'].value_counts().sort_index())
print(f'\nQS moyen : {keywords["quality_score"].mean():.2f}')


### 💡 INTERPRÉTATION — Premières observations

**Sur la performance quotidienne :**
- Le mix **device** révèle la répartition du trafic — un Paid Media Manager doit y prêter
  attention car Mobile et Desktop ont des comportements très différents (taux de conversion,
  avg position)
- La distribution **conversion_type** montre le mix de résultats — en B2B on attend plus de
  Leads/Demo, en e-commerce plus de Purchase

**Sur les campagnes :**
- La distribution par **campaign_type** indique la stratégie de l'agence — dominance Search
  = stratégie intent-based, dominance PMax = confiance dans l'automation Google
- Les campagnes **Paused** ou **Removed** doivent être gérées avec précaution (ne pas les
  inclure dans les KPIs actuels)

**Sur les keywords :**
- Le **Quality Score moyen** est un indicateur clé : < 5 = campagnes coûteuses et inefficaces,
  > 7 = campagnes bien optimisées
- La distribution **match_type** révèle la philosophie : Exact/Phrase = contrôle, Broad = volume


---
## 5. Diagnostic qualité

### 🎓 MÉTHODE — Audit en entonnoir

```
1. Structure (shape, types)       → Forme attendue ?
   ▼
2. Nulls                          → Qui manque, est-ce légitime ?
   ▼
3. Doublons                       → Clé primaire unique ?
   ▼
4. Anomalies techniques           → Valeurs impossibles (cost<0, clicks>imp)
   ▼
5. Incohérences métier            → QS hors plage, conversions>clicks
```


In [ ]:
print('━' * 60)
print('  AUDIT QUALITÉ — GoogleAdsPulse')
print('━' * 60)

# ── 1. NULLS par table ───────────────────────────────────────────────
print('\n📋 1. VALEURS MANQUANTES')
print('-' * 60)

# Sur perf, keyword_id PEUT être NULL (Display/Video/PMax = automated)
# Les autres IDs doivent TOUJOURS être présents
critiques = {
    'campaigns':  ['campaign_id', 'account_id', 'campaign_type', 'statut'],
    'ads':        ['ad_id', 'campaign_id', 'quality_score'],
    'keywords':   ['keyword_id', 'campaign_id', 'quality_score'],
    'perf':       ['date', 'account_id', 'campaign_id', 'device', 'impressions', 'clicks', 'cost_eur'],
}

tables = {'campaigns':campaigns,'ads':ads,'keywords':keywords,'perf':perf}
for tname, cols in critiques.items():
    df = tables[tname]
    print(f'\n  {tname} ({len(df):,} lignes) :')
    for col in cols:
        if col in df.columns:
            n = df[col].isnull().sum()
            pct = n / len(df) * 100 if len(df) > 0 else 0
            flag = '✅' if n == 0 else '⚠️'
            print(f'    {flag} {col:<22} : {n:>6,} nulls ({pct:>4.1f}%)')

# Note : keyword_id NULL légitime sur Display/Video/PMax
n_kw_null = perf['keyword_id'].isnull().sum()
print(f'\n  ℹ️  perf.keyword_id NULL : {n_kw_null:,} ({n_kw_null/len(perf)*100:.1f}%) — normal pour Display/Video/PMax (automated)')


In [ ]:
# ── 2. DOUBLONS ───────────────────────────────────────────────────────
print('\n📋 2. DOUBLONS (clés primaires doivent être uniques)')
print('-' * 60)

pk_check = [('campaigns','campaign_id'),('ads','ad_id'),('keywords','keyword_id'),('accounts','account_id')]
tables['accounts'] = accounts

for tname, pk in pk_check:
    df = tables[tname]
    if pk in df.columns:
        n_dup = df[pk].duplicated().sum()
        flag = '✅' if n_dup == 0 else '⚠️'
        print(f'  {flag} {tname:<15} {pk:<18} : {n_dup} doublons')

# La table perf n'a pas de clé primaire simple — on regarde la clé composite
perf_pk = ['date','campaign_id','ad_id','keyword_id','device','country','hour']
n_dup_perf = perf.duplicated(subset=perf_pk).sum()
flag = '✅' if n_dup_perf == 0 else '⚠️'
print(f'  {flag} perf (date+campaign+ad+keyword+device+country+hour) : {n_dup_perf} doublons')


In [ ]:
# ── 3. ANOMALIES TECHNIQUES ──────────────────────────────────────────
print('\n📋 3. ANOMALIES TECHNIQUES (valeurs impossibles)')
print('-' * 60)

# 3.1 Cost négatif (bug refund mal codé)
n_neg = (perf['cost_eur'] < 0).sum()
print(f'  {"⚠️" if n_neg else "✅"} Cost_eur NÉGATIF                 : {n_neg}')

# 3.2 Clicks > impressions (impossible logiquement)
n_cli = (perf['clicks'] > perf['impressions']).sum()
print(f'  {"⚠️" if n_cli else "✅"} Clicks > Impressions             : {n_cli}')

# 3.3 Conversions > clicks (rare mais possible en multi-touch)
n_conv = (perf['conversions'] > perf['clicks']).sum()
print(f'  {"⚠️" if n_conv else "✅"} Conversions > Clicks             : {n_conv}')

# 3.4 Conversions > 0 mais clicks = 0 (impossible)
n_imp = ((perf['conversions'] > 0) & (perf['clicks'] == 0)).sum()
print(f'  {"⚠️" if n_imp else "✅"} Conversions > 0 avec 0 clicks    : {n_imp}')

# 3.5 Quality Score hors plage [1-10]
qs_ads = ((ads['quality_score'] < 1) | (ads['quality_score'] > 10)).sum()
qs_kw  = ((keywords['quality_score'] < 1) | (keywords['quality_score'] > 10)).sum()
print(f'  {"⚠️" if qs_ads else "✅"} QS hors plage [1-10] sur ads     : {qs_ads}')
print(f'  {"⚠️" if qs_kw else "✅"} QS hors plage [1-10] sur keywords : {qs_kw}')


In [ ]:
# ── 4. INCOHÉRENCES MÉTIER ───────────────────────────────────────────
print('\n📋 4. INCOHÉRENCES MÉTIER')
print('-' * 60)

# 4.1 Ads orphelins : campaign_id n'existe pas dans campaigns
camp_ids = set(campaigns['campaign_id'])
n_ad_orph = (~ads['campaign_id'].isin(camp_ids)).sum()
print(f'  {"⚠️" if n_ad_orph else "✅"} Ads orphelins (campaign absente) : {n_ad_orph}')

# 4.2 Keywords orphelins
n_kw_orph = (~keywords['campaign_id'].isin(camp_ids)).sum()
print(f'  {"⚠️" if n_kw_orph else "✅"} Keywords orphelins               : {n_kw_orph}')

# 4.3 Performance sans campaign parent
n_perf_orph = (~perf['campaign_id'].isin(camp_ids)).sum()
print(f'  {"⚠️" if n_perf_orph else "✅"} Perf orphelines (campaign absente): {n_perf_orph}')

# 4.4 Clicks mais zéro cost (suspect sauf si campagne gratuite — n'existe pas en Google Ads)
n_free = ((perf['clicks'] > 0) & (perf['cost_eur'] == 0)).sum()
print(f'  {"⚠️" if n_free else "✅"} Clicks > 0 avec 0 EUR de cost    : {n_free}')


### 💡 INTERPRÉTATION — Actions à prendre

**Checklist de corrections à appliquer dans le pipeline (sans modifier le dataset source) :**

| Anomalie | Action |
|---|---|
| Doublons performance | `drop_duplicates(subset=[date,campaign,ad,keyword,device,country,hour])` |
| Cost négatifs | Exclusion des KPIs agrégés (ou imputation à 0 si volume faible) |
| Clicks > impressions | Exclusion — c'est un bug d'attribution |
| Conversions > clicks | À flagger mais conserver — possible en multi-touch |
| QS hors plage [1-10] | Remplacer par NULL ou capper à [1,10] |
| Keywords orphelins | Exclusion (pas rattachable à une campaign active) |

### 🏢 ACTION MÉTIER — Remontée à Marc-Aurèle

> *« Marc-Aurèle, avant de te présenter les chiffres, quelques observations qualité :*
>
> *1. Nous avons détecté ~40 doublons sur la table performance — probablement un problème
> d'export Google Ads API (réimport d'une journée déjà chargée). À remonter à l'équipe
> ops pour ajuster le pipeline ETL.*
>
> *2. Quelques coûts négatifs (~10) — sûrement des refunds mal codés côté facturation
> Google Ads. Impact marginal sur le total mais à clarifier avec ton contact Google.*
>
> *3. Quelques Quality Scores hors plage [1-10] — ce sont probablement des valeurs par
> défaut (0 ou NULL) mal converties à l'import.*
>
> *Ces anomalies représentent moins de 0.2% du volume — nos KPIs restent robustes, mais
> il faut les remonter pour fiabiliser le reporting futur. »*


---
## 6. Les 10 KPIs Coupler.io

### 🎓 MÉTHODE — Pourquoi ces 10 KPIs ?

Ces 10 indicateurs forment la **vue Overview standard** d'un dashboard Google Ads
(reproduit par tous les outils pros : Coupler.io, Supermetrics, Looker Studio).
Ils se lisent en 3 groupes :

**Groupe 1 — Volume** (à quel volume tournons-nous ?)
| # | KPI | Calcul |
|---|---|---|
| 1 | **Impressions** | Nb de fois où l'annonce a été affichée |
| 2 | **Clicks** | Nb de clics reçus |
| 3 | **Conversions** | Nb d'actions valorisées (purchase, lead, etc.) |

**Groupe 2 — Efficacité** (à quel coût ?)
| # | KPI | Calcul | Règle |
|---|---|---|---|
| 4 | **CTR** | clicks / impressions × 100 | Plus élevé = mieux |
| 5 | **CPC** | cost / clicks | Plus bas = mieux |
| 6 | **CPM** | cost / impressions × 1000 | Plus bas = mieux |
| 7 | **Cost/Conversion** | cost / conversions | Plus bas = mieux |
| 8 | **Conversion rate** | conversions / clicks × 100 | Plus élevé = mieux |

**Groupe 3 — Rentabilité** (est-ce que ça rapporte ?)
| # | KPI | Calcul |
|---|---|---|
| 9 | **Spend amount** | somme des cost_eur |
| 10 | **Conversions value** | somme des conversion_value_eur |
| Bonus | **ROAS** | conversions value / spend (doit être > 3 pour rentabilité) |

> **⚠️ On reste sur des KPIs GLOBAUX.** Les deltas period-over-period, Top/Bottom,
> breakdowns et cohort analysis sont traités dans le **NB2** avec SQL.


In [ ]:
# Nettoyage minimal pour des KPIs fiables
perf_clean = (perf
    .drop_duplicates(subset=['date','campaign_id','ad_id','keyword_id','device','country','hour'])
    .query('cost_eur >= 0')                          # Exclure cost négatifs
    .query('clicks <= impressions')                  # Exclure clicks > impressions
    .copy()
)

print(f'Volume avant nettoyage : {len(perf):,} lignes')
print(f'Volume après nettoyage : {len(perf_clean):,} lignes')
print(f'Volume exclu           : {len(perf)-len(perf_clean):,} lignes ({(1-len(perf_clean)/len(perf))*100:.2f}%)')


In [ ]:
# ── CALCUL DES 10 KPIs COUPLER.IO ─────────────────────────────────────

# Agrégats de base
total_impressions   = perf_clean['impressions'].sum()
total_clicks        = perf_clean['clicks'].sum()
total_conversions   = perf_clean['conversions'].sum()
total_spend         = perf_clean['cost_eur'].sum()
total_conv_value    = perf_clean['conversion_value_eur'].sum()

# KPIs calculés
ctr              = total_clicks / total_impressions * 100       if total_impressions else 0
cpc              = total_spend / total_clicks                     if total_clicks else 0
cpm              = total_spend / total_impressions * 1000         if total_impressions else 0
cost_per_conv    = total_spend / total_conversions                if total_conversions else 0
conv_rate        = total_conversions / total_clicks * 100         if total_clicks else 0
roas             = total_conv_value / total_spend                 if total_spend else 0

# Période
periode_debut = perf_clean['date'].min().date()
periode_fin   = perf_clean['date'].max().date()
nb_jours      = (perf_clean['date'].max() - perf_clean['date'].min()).days + 1

# Affichage style Coupler.io
print('═' * 70)
print('  GOOGLEADSPULSE — OVERVIEW (style Coupler.io)')
print('═' * 70)
print(f'  Période : {periode_debut} → {periode_fin} ({nb_jours} jours)')
print(f'  Accounts : {len(accounts)} | Campaigns : {len(campaigns)}')
print('─' * 70)
print(f'  Groupe 1 — VOLUME')
print(f'    1. Impressions       : {total_impressions:>18,}')
print(f'    2. Clicks            : {total_clicks:>18,}')
print(f'    3. Conversions       : {total_conversions:>18,}')
print('─' * 70)
print(f'  Groupe 2 — EFFICACITÉ')
print(f'    4. CTR               : {ctr:>17.2f}%')
print(f'    5. CPC               : {cpc:>17.2f} EUR')
print(f'    6. CPM               : {cpm:>17.2f} EUR')
print(f'    7. Cost/Conversion   : {cost_per_conv:>17.2f} EUR')
print(f'    8. Conversion rate   : {conv_rate:>17.2f}%')
print('─' * 70)
print(f'  Groupe 3 — RENTABILITÉ')
print(f'    9. Spend amount      : {total_spend:>17,.0f} EUR')
print(f'   10. Conversions value : {total_conv_value:>17,.0f} EUR')
print(f'    B. ROAS              : {roas:>17.2f} ×')
print('═' * 70)


### 💡 INTERPRÉTATION — Lecture des 10 KPIs

**KPI 4 — CTR (~2.5% à 3%)**
Le CTR global mélange Search (élevé), Display (très bas) et PMax (moyen).
Un CTR global > 2% est correct ; < 1% = problème de ciblage généralisé.
Le vrai pilotage se fait au niveau de chaque type de campagne (voir NB2).

**KPI 5 — CPC (~0.40 EUR)**
CPC moyen qui reflète la compétitivité du marché. Sur des marchés africains,
un CPC < 0.50 EUR est normal ; sur des mots-clés très concurrentiels (finance, assurance),
le CPC peut dépasser 2 EUR.

**KPI 7 — Cost/Conversion (~10 EUR)**
**C'est LE KPI à regarder en priorité.** Un Cost/Conv de 10 EUR pour un SaaS B2B est excellent,
pour un e-commerce à 20 EUR de marge c'est limite, pour un lead immobilier à 500 EUR de marge
c'est un steal. Le CPC seul ne veut rien dire sans le contexte du revenu.

**KPI 8 — Conversion rate (~4%)**
Taux de conversion global. 4% est bon pour du B2B ciblé, moyen pour de l'e-commerce
(qui tourne plus autour de 2%). À décomposer par device et campaign_type dans le NB2.

**Bonus — ROAS (~6×)**
**Le verdict final de la rentabilité.** 1 EUR dépensé génère 6 EUR de revenu.
Règle d'arbitrage : **ROAS > 3 = rentable**, **ROAS > 5 = très rentable**.

### 🏢 ACTION MÉTIER — Le cockpit du lundi matin pour Marc-Aurèle

Ces 10 KPIs remplacent la **première heure** de son reporting hebdomadaire :

> *« Marc-Aurèle, en un coup d'œil tu vois que sur les 24 derniers mois :*
> - *56M d'impressions, 1.5M de clics, 60k conversions*
> - *Ton ROAS global est de 6× — très rentable*
> - *Ton Cost/Conv de 10 EUR est bien calibré pour tes objectifs clients*
>
> *Les vraies questions opérationnelles (quelle campagne performe, où sont les anomalies,
> quels keywords s'essoufflent) arrivent dans le NB2. »*


---
## 7. Visualisation Overview 2×2 (style Coupler.io)

### 🎓 MÉTHODE — Reproduire la page Overview

| Cadran | Graphique Coupler.io | Type |
|---|---|---|
| Haut-gauche | Impressions vs Clicks | Dual-axis timeline |
| Haut-droit | Clicks vs CPC | Dual-axis timeline |
| Bas-gauche | Spend amount by Date | Single-line timeline |
| Bas-droit | Conversions vs Conversion rate | Dual-axis timeline |

Les 4 graphiques utilisent l'agrégation **quotidienne** lissée sur 7 jours pour la lisibilité.


In [ ]:
# Agrégation quotidienne pour les 4 timelines
daily = (perf_clean
    .groupby('date')
    .agg(impressions=('impressions','sum'),
         clicks=('clicks','sum'),
         cost=('cost_eur','sum'),
         conversions=('conversions','sum'))
    .reset_index()
)
daily['cpc']       = daily['cost'] / daily['clicks'].replace(0, np.nan)
daily['conv_rate'] = daily['conversions'] / daily['clicks'].replace(0, np.nan) * 100

# Lissage moyenne mobile 7 jours pour la lisibilité
for col in ['impressions','clicks','cost','conversions','cpc','conv_rate']:
    daily[f'{col}_ma7'] = daily[col].rolling(window=7, min_periods=1).mean()

print(f'✅ Série temporelle : {len(daily)} jours')
print(daily.head(3))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('GoogleAdsPulse — Overview (style Coupler.io)', fontsize=15, fontweight='bold', y=0.995)

# ══ Cadran 1 : Impressions vs Clicks ═══════════════════════════════════
ax1 = axes[0, 0]
ax1b = ax1.twinx()
ax1.plot(daily['date'], daily['impressions_ma7'], color=COLORS['primary'], linewidth=2, label='Impressions (MA7)')
ax1b.plot(daily['date'], daily['clicks_ma7'],     color=COLORS['blue'],    linewidth=2, label='Clicks (MA7)')
ax1.set_title('Impressions vs Clicks', fontweight='bold')
ax1.set_ylabel('Impressions', color=COLORS['primary'])
ax1b.set_ylabel('Clicks', color=COLORS['blue'])
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f'{int(v/1000)}K'))
ax1b.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f'{int(v/1000)}K'))
ax1.tick_params(axis='x', rotation=30)

# ══ Cadran 2 : Clicks vs CPC ═══════════════════════════════════════════
ax2 = axes[0, 1]
ax2b = ax2.twinx()
ax2.plot(daily['date'], daily['clicks_ma7'], color=COLORS['blue'],    linewidth=2, label='Clicks')
ax2b.plot(daily['date'], daily['cpc_ma7'],   color=COLORS['warning'], linewidth=2, label='CPC')
ax2.set_title('Clicks vs CPC', fontweight='bold')
ax2.set_ylabel('Clicks', color=COLORS['blue'])
ax2b.set_ylabel('CPC (EUR)', color=COLORS['warning'])
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f'{int(v/1000)}K'))
ax2.tick_params(axis='x', rotation=30)

# ══ Cadran 3 : Spend amount by Date ════════════════════════════════════
ax3 = axes[1, 0]
ax3.plot(daily['date'], daily['cost_ma7'], color=COLORS['danger'], linewidth=2)
ax3.fill_between(daily['date'], daily['cost_ma7'], alpha=0.12, color=COLORS['danger'])
ax3.set_title('Spend amount by Date', fontweight='bold')
ax3.set_ylabel('Spend (EUR)')
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f'{int(v):,}'))
ax3.tick_params(axis='x', rotation=30)

# ══ Cadran 4 : Conversions vs Conversion rate ══════════════════════════
ax4 = axes[1, 1]
ax4b = ax4.twinx()
ax4.plot(daily['date'], daily['conversions_ma7'], color=COLORS['secondary'], linewidth=2)
ax4b.plot(daily['date'], daily['conv_rate_ma7'],  color='#9333EA',            linewidth=2)
ax4.set_title('Conversions vs Conversion rate', fontweight='bold')
ax4.set_ylabel('Conversions', color=COLORS['secondary'])
ax4b.set_ylabel('Conversion rate (%)', color='#9333EA')
ax4.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}googleadspulse_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure sauvegardée : {SAVE_PATH}googleadspulse_overview.png')


### 💡 INTERPRÉTATION — Lecture des 4 timelines

**Cadran 1 — Impressions vs Clicks**
Les deux courbes doivent **évoluer en parallèle**. Si impressions montent mais clicks
restent plats → problème de CTR (pubs qui s'affichent mais n'intéressent plus). 
Si clicks montent plus que impressions → CTR s'améliore (optimisation ads ou ciblage).

**Cadran 2 — Clicks vs CPC**
**Le graphique le plus révélateur pour Marc-Aurèle.** Si le CPC monte en même temps que
les clicks, c'est qu'on achète du trafic plus cher mais qu'on en obtient plus
(expansion budget). Si le CPC monte et les clicks stagnent → problème : on paye plus
pour la même chose. **Pattern saisonnier** visible : le CPC explose en novembre-décembre
(Black Friday + Noël) et en janvier (reset budgets).

**Cadran 3 — Spend by Date**
La courbe doit suivre approximativement les cycles business. Les **pics anormaux** sont
à investiguer — c'est précisément ce que le NB2 automatisera avec un z-score.

**Cadran 4 — Conversions vs Conversion rate**
Un volume de conversions qui monte avec un conv rate qui baisse = on scale mais on perd
en qualité. Un conv rate stable avec des conversions en hausse = scaling sain.

### 🏢 ACTION MÉTIER — Les 3 questions qui resteront pour le NB2

1. **Quelle campagne** est responsable des pics de spend visibles sur le cadran 3 ?
2. **Quelle évolution Week-over-Week** sur chaque KPI (les deltas `+3.41%` / `-11.42%` du dashboard) ?
3. **Quels keywords** perdent en performance sur la durée (cohort analysis) ?

Ces 3 questions sont traitées dans le NB2 avec SQL avancé.


---
## 8. Bilan du Notebook 1

### ✅ Ce qui a été réalisé

| Étape | Livrable |
|---|---|
| Brief métier | 3 questions de Marc-Aurèle traduites en questions analytiques |
| Dictionnaire | 5 tables documentées + schéma étoile |
| Chargement | 5 DataFrames pandas avec `parse_dates` |
| Exploration | `.head()`, `.info()`, `.describe()`, `.value_counts()` sur les 3 tables clés |
| Diagnostic | Nulls, doublons, anomalies techniques, incohérences métier |
| 10 KPIs Coupler.io | Volume + Efficacité + Rentabilité |
| Visualisation | Figure 2×2 Overview (style Coupler.io) |

### 📋 Checklist pour le NB2

```
☐  Réutiliser perf_clean (dédoublonné + filtré)
☐  Calculer les 10 KPIs EN SQL avec deltas period-over-period (LAG)
☐  Top & Bottom campaigns avec RANK + NTILE
☐  Rolling 7-day average sur les KPIs clés (SUM OVER ROWS BETWEEN)
☐  Cohort analysis des keywords (DATE_TRUNC + DATEDIFF + pivot)
☐  Détection anomalies par z-score (AVG OVER + STDDEV OVER)
☐  Breakdown device × jour × heure (pivots SQL)
☐  Self-join campagne vs benchmark account
☐  Export des 7 CSV analytiques pour Power BI
```

### 🧭 Progression pédagogique

```
NB1 ──► Contexte + 10 KPIs Overview (ce notebook)
 │
 ▼
NB2 ──► SQL avancé : window functions + cohort analysis + z-score
 │
 ▼
NB3 ──► Power BI : 5 pages dashboard style Coupler.io + 30+ mesures DAX
```


---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.
